# Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks

**Paper**: Ren et al., NIPS 2015

## The Last Bottleneck: Selective Search

Fast R-CNN reduced CNN computation to one pass, but **Selective Search** — a CPU algorithm — still took ~2 seconds per image and couldn't be trained end-to-end.

## Faster R-CNN's Solution: Region Proposal Network (RPN)

<img src='../figures/faster_rcnn_pipeline.png' width='650'/>

Faster R-CNN introduces a **Region Proposal Network (RPN)** that:
- Shares the same convolutional backbone with the detection head (nearly free!)
- Slides a small network over the feature map to propose boxes at multiple scales/ratios
- Outputs ~300 high-quality proposals vs Selective Search's ~2000 low-quality ones
- Is fully differentiable → trained end-to-end with the detector

**Total result**: ~0.2 sec/image — real-time capable on GPU.

## Anchor Boxes

At each location of the feature map, the RPN considers **k anchor boxes** at different scales and aspect ratios (e.g., k=9: 3 scales x 3 ratios). For each anchor:
- **Objectness score**: is this anchor likely to contain an object? (binary classification)
- **Box offsets**: refine the anchor to better fit the object

Anchors with IoU > 0.7 with any ground-truth box → positive. IoU < 0.3 → negative.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.ops import box_iou, nms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import requests
from io import BytesIO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Generate Anchor Boxes

For a feature map of size H×W with stride s, we place k anchors at each of the H×W locations → total H×W×k anchors. For a 600×1000 image with stride 16: 37×63×9 = ~21,000 anchors.

In [ ]:
def generate_anchors(feat_h, feat_w, stride, scales, ratios):
    """
    Generate anchor boxes for the full feature map.
    Returns: (feat_h * feat_w * len(scales)*len(ratios), 4) in [x1,y1,x2,y2] image coords.
    """
    anchors = []
    for y in range(feat_h):
        for x in range(feat_w):
            cx = (x + 0.5) * stride  # center in image space
            cy = (y + 0.5) * stride
            for s in scales:
                area = s * s
                for r in ratios:
                    w = np.sqrt(area / r)
                    h = w * r
                    anchors.append([cx - w/2, cy - h/2, cx + w/2, cy + h/2])
    return np.array(anchors, dtype=np.float32)

# Example: 8x8 feature map, stride=32, 3 scales x 3 ratios
anchors = generate_anchors(8, 8, stride=32,
                            scales=[64, 128, 256],
                            ratios=[0.5, 1.0, 2.0])
print(f'Anchors shape: {anchors.shape}  ({8*8*9} anchors = 8x8 feat x 9 per location)')
print(f'First anchor:  {anchors[0].round(1)}')

# Visualize anchors at one location
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
img_dummy = np.ones((256, 256, 3)) * 0.9
ax.imshow(img_dummy)
# Show 9 anchors at center of image (location [3,3])
center_idx = (3*8 + 3) * 9
colors_anch = plt.cm.tab10(np.linspace(0,1,9))
scales_l=['64','128','256']; ratios_l=['0.5','1.0','2.0']
for i, (anch, c) in enumerate(zip(anchors[center_idx:center_idx+9], colors_anch)):
    x1,y1,x2,y2 = np.clip(anch, 0, 255)
    s_i, r_i = i//3, i%3
    ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=2,
        edgecolor=c,facecolor='none',label=f's={scales_l[s_i]},r={ratios_l[r_i]}'))
ax.set_xlim(0,255); ax.set_ylim(255,0)
ax.set_title('9 anchors at one feature map location (3 scales x 3 ratios)')
ax.legend(loc='upper right',fontsize=7); plt.tight_layout(); plt.show()

## Region Proposal Network (RPN)

The RPN is a tiny network that slides over the feature map:
1. 3x3 conv → intermediate features
2. Two 1x1 conv branches:
   - **cls branch**: 2k scores (object/background per anchor)
   - **reg branch**: 4k offsets (dx, dy, dw, dh per anchor)

In [ ]:
class RPN(nn.Module):
    def __init__(self, in_channels=512, mid_channels=512, n_anchors=9):
        super().__init__()
        self.conv    = nn.Conv2d(in_channels, mid_channels, 3, padding=1)
        self.cls_out = nn.Conv2d(mid_channels, n_anchors * 2, 1)   # 2=obj/bg
        self.reg_out = nn.Conv2d(mid_channels, n_anchors * 4, 1)   # 4=dx,dy,dw,dh

        nn.init.normal_(self.conv.weight, 0, 0.01)
        nn.init.normal_(self.cls_out.weight, 0, 0.01)
        nn.init.normal_(self.reg_out.weight, 0, 0.01)

    def forward(self, feat):
        B, C, H, W = feat.shape
        h    = F.relu(self.conv(feat))
        cls  = self.cls_out(h)  # (B, 2k, H, W)
        reg  = self.reg_out(h)  # (B, 4k, H, W)
        # Reshape: (B, H, W, k, 2/4) → (B*H*W*k, 2/4)
        cls = cls.permute(0,2,3,1).reshape(-1, 2)
        reg = reg.permute(0,2,3,1).reshape(-1, 4)
        return cls, reg

rpn = RPN()
dummy_feat = torch.randn(1, 512, 38, 56)  # ~600x900 image with stride 16
cls_out, reg_out = rpn(dummy_feat)
print(f'Feature map: {tuple(dummy_feat.shape)}')
print(f'RPN cls:     {tuple(cls_out.shape)}  ({38*56*9} anchors x 2 scores)')
print(f'RPN reg:     {tuple(reg_out.shape)}  ({38*56*9} anchors x 4 offsets)')

## Full Faster R-CNN Inference with torchvision

torchvision provides a complete Faster R-CNN with ResNet50 + FPN backbone, pre-trained on COCO (80 classes).

In [ ]:
COCO_NAMES = [
    '__background__','person','bicycle','car','motorcycle','airplane','bus','train',
    'truck','boat','traffic light','fire hydrant','stop sign','parking meter','bench',
    'bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe',
    'backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard',
    'sports ball','kite','baseball bat','baseball glove','skateboard','surfboard',
    'tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl',
    'banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza',
    'donut','cake','chair','couch','potted plant','bed','dining table','toilet',
    'tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven',
    'toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear',
    'hair drier','toothbrush'
]

model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT).to(device).eval()

try:
    url = 'https://ultralytics.com/images/bus.jpg'
    img_pil = Image.open(BytesIO(requests.get(url, timeout=8).content)).convert('RGB')
except Exception:
    img_pil = Image.fromarray(np.random.randint(0,255,(480,640,3),dtype=np.uint8))

img_t = T.ToTensor()(img_pil).to(device)
with torch.no_grad():
    pred = model([img_t])[0]

keep   = pred['scores'] > 0.5
boxes  = pred['boxes'][keep].cpu().numpy()
labels = pred['labels'][keep].cpu().numpy()
scores = pred['scores'][keep].cpu().numpy()

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img_pil)
cmap = plt.cm.Set1(np.linspace(0, 1, max(len(boxes),1)))
for box, lbl, sc, c in zip(boxes, labels, scores, cmap):
    x1,y1,x2,y2 = box
    ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=2,edgecolor=c,facecolor='none'))
    ax.text(x1,max(0,y1-5), f'{COCO_NAMES[lbl]} {sc:.2f}',
            color='white', fontsize=8, fontweight='bold',
            bbox=dict(facecolor=c, alpha=0.7, pad=1))
ax.set_title(f'Faster R-CNN (ResNet50+FPN, COCO) — {len(boxes)} detections')
ax.axis('off'); plt.tight_layout(); plt.show()

## Fine-tune Faster R-CNN on Custom Data

torchvision's Faster R-CNN is easy to fine-tune. Just replace the box predictor head with the desired number of classes.

In [ ]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_fasterrcnn_model(num_classes):
    """Load pretrained Faster R-CNN and replace the head for num_classes."""
    model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


# Example: 2-class detector (background + cat)
custom_model = get_fasterrcnn_model(num_classes=2).to(device)
custom_model.train()

# Dummy batch (as torchvision expects)
dummy_imgs   = [torch.randn(3, 300, 400).to(device)]
dummy_targets = [{
    'boxes':  torch.tensor([[50.,50.,150.,200.]], dtype=torch.float32).to(device),
    'labels': torch.tensor([1], dtype=torch.int64).to(device),
}]

loss_dict = custom_model(dummy_imgs, dummy_targets)
total_loss = sum(loss_dict.values())
print('Loss components:')
for k, v in loss_dict.items():
    print(f'  {k}: {v.item():.4f}')
print(f'Total loss: {total_loss.item():.4f}')
print('Ready to backprop!')

## Faster R-CNN Architecture Summary

```
Image
  └─ Backbone (ResNet50+FPN) ──────────────── shared feature pyramid
       ├─ RPN head
       │    ├─ Objectness scores  (2k per location)
       │    └─ Box offsets        (4k per location)
       │    → NMS → ~300 proposals
       └─ RoI Align (proposals projected onto feature map)
            └─ Detection head
                 ├─ Class scores  (N_classes+1)
                 └─ Box offsets   (4 * N_classes)
```

| Component | Detail |
|-----------|--------|
| **Backbone** | ResNet50 + FPN (multi-scale features P2–P6) |
| **RPN anchors** | 3 scales x 3 ratios = 9 per location |
| **Proposals** | ~2000 from RPN → top 300 after NMS |
| **RoI Align** | 7×7 fixed output (replaces quantizing RoI Pool) |
| **Loss** | RPN: binary CE + smooth-L1; Head: CE + smooth-L1 |
| **Speed** | ~0.2 sec/image (vs R-CNN 47 sec) |
| **mAP COCO** | ~37 AP (ResNet50+FPN) |